In [2]:
import requests
from bs4 import BeautifulSoup
import feedparser
import os
import re
import time


In [3]:
feed_url = "https://shortfictionbreak.com/feed/"
feed = feedparser.parse(feed_url)
urls = [entry.link for entry in feed.entries]
print(f"Found {len(urls)} story URLs")
urls[:5]  # preview a few

Found 10 story URLs


['https://shortfictionbreak.com/the-weight-of-silence/',
 'https://shortfictionbreak.com/king-and-lionheart/',
 'https://shortfictionbreak.com/the-whisperer/',
 'https://shortfictionbreak.com/fortune-and-glory/',
 'https://shortfictionbreak.com/the-golden-seed/']

In [4]:
def clean_text(text):
    """Basic whitespace + dash cleanup."""
    text = re.sub(r"\s+", " ", text)
    text = text.replace("–", "-").strip()
    return text

def scrape_story(url, out_dir="stories"):
    """Download one story from Short Fiction Break."""
    try:
        r = requests.get(url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")

        title_tag = soup.find("h1", class_="entry-title")
        content_tag = soup.find("div", class_="entry-content")

        if not content_tag:
            print(f"No content found: {url}")
            return None

        title = title_tag.get_text(strip=True) if title_tag else "untitled"
        story = clean_text(content_tag.get_text(separator="\n", strip=True))

        os.makedirs(out_dir, exist_ok=True)
        filename = re.sub(r"[^a-zA-Z0-9_-]", "_", title)[:80]

        with open(f"{out_dir}/{filename}.txt", "w", encoding="utf-8") as f:
            f.write(story)

        print(f"Saved: {filename}.txt")
        return filename

    except Exception as e:
        print(f"Error scraping {url}: {e}")

In [5]:
for url in urls[:3]:
    scrape_story(url)
    time.sleep(1)


Saved: The_Weight_of_Silence.txt
Saved: King_and_Lionheart.txt
Saved: The_Whisperer.txt


In [6]:
import csv

with open("stories_metadata.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["title", "url"])
    for entry in feed.entries:
        writer.writerow([entry.title, entry.link])

print("Saved metadata.csv")


Saved metadata.csv
